# T3.7 — Multi-Script Handwritten Indic Character Recognition

**Pipeline:** Script Router (4-class CNN) → Per-Script Character CNN  
**Scripts:** Devanagari (46 cls) · Tamil (156 cls) · Bengali (84 cls) · Telugu (6 cls)  

---
## Table of Contents
1. Packages
2. Run-mode switch (`KAGGLE_SMOKE`)
3. Imports from `src/`
4. Dataset paths
5. Exploratory Data Analysis
6. Stage 1 — Router training
7. Stage 2 — Character classifier training
8. Evaluation
9. Ablation studies
10. Results export


In [4]:
# ── 1. Install missing packages ─────────────────────────────────────────────
import subprocess, sys
_pkgs = ['scikit-learn', 'seaborn', 'tqdm', 'matplotlib', 'pandas', 'Pillow']
for _p in _pkgs:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', _p],
                   capture_output=True)
print('packages ready')


packages ready


In [ ]:
# ── 2. Run-mode switch ───────────────────────────────────────────────────────
#
#  KAGGLE_SMOKE_RUN = True   → quick validation (~15 min on 2×T4)
#                              3 epochs, 50 imgs/class, no ablations
#  KAGGLE_SMOKE_RUN = False  → full training run (20/30 epochs, all data)
#
#  Must be set BEFORE any src/ import so config.py picks it up.
import os
KAGGLE_SMOKE_RUN = False    # <── flip to False for the full run

if KAGGLE_SMOKE_RUN:
    os.environ['KAGGLE_SMOKE'] = '1'
else:
    os.environ.pop('KAGGLE_SMOKE', None)

print(f'KAGGLE_SMOKE_RUN = {KAGGLE_SMOKE_RUN}')


KAGGLE_SMOKE_RUN = False


In [6]:
# ── 3. Imports from src/ ─────────────────────────────────────────────────────
# src/*.py are uploaded as Kaggle dataset dhruv10050/indic-handwriting-src,
# mounted at /kaggle/input/indic-handwriting-src/ on the kernel machine.
# Locally they live in ./src/.
import os, sys, warnings, json, random, zipfile
from pathlib import Path
warnings.filterwarnings('ignore')

def _add_src():
    candidates = [
        '/kaggle/input/indic-handwriting-src',  # dataset mount (Kaggle, primary)
        str(Path(os.getcwd()) / 'src'),          # local: ./src
        './src',
    ]
    # Debug: show what Kaggle has mounted
    ki = Path('/kaggle/input')
    if ki.exists():
        print('Available at /kaggle/input:', [d.name for d in sorted(ki.iterdir())])
    for p in candidates:
        if p and Path(p).is_dir():
            if p not in sys.path:
                sys.path.insert(0, p)
            print(f'src/ loaded from: {p}')
            return
    # Hard fail with diagnostic info
    raise RuntimeError(
        f'Cannot locate src/ modules.\n'
        f'  CWD          : {os.getcwd()}\n'
        f'  CWD contents : {os.listdir(os.getcwd())[:30]}\n'
        f'  sys.path     : {sys.path[:8]}'
    )

_add_src()

from config import (
    ENV, DEVICE, NUM_GPUS, TEST_RUN, KAGGLE_SMOKE, RUN_ABLATIONS,
    IMG_SIZE, ROUTER_EPOCHS, CHAR_EPOCHS, ABLATION_EPOCHS,
    PATIENCE, ROUTER_BATCH, CHAR_BATCH, TELUGU_BATCH, CHAR_MAX,
    CKPT_DIR, LOG_DIR, RES_DIR, FIG_DIR, WORK_DIR,
    NORM_MEAN, NORM_STD, print_config,
)
from utils      import seed_everything, free_memory, wrap_model, unwrap, find_path
from transforms import get_transforms
from datasets   import (
    ScriptImageDataset, BanglaLekhaDataset, TeluguCSVDataset,
    make_loader, weighted_sampler, build_router_loaders,
)
from models     import ScriptRouter, ScriptCNN, TwoStagePipeline
from trainer    import (train_epoch, validate, train_model,
                        load_checkpoint, plot_history)
from evaluate   import (topk_acc, eval_script, plot_confusion,
                        run_robustness_eval, plot_robustness,
                        run_latency_benchmark)
from ablations  import run_all_ablations, plot_ablation_results

import torch
import torch.nn as nn
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import classification_report, confusion_matrix

seed_everything()
print_config()


src/ loaded from: /Users/dhruvrajput/sem-2-assignments/SMAI/A3/src


SyntaxError: '[' was never closed (ablations.py, line 134)

## 4. Dataset Paths

Add the following Kaggle datasets via **+ Add Data**:
| Slug | Script |
|------|--------|
| `medahmedkrichen/devanagari-handwritten-character-datase` | Devanagari |
| `faizalhajamohideen/uthcdtamil-handwritten-database` | Tamil |
| `syamkakarla/telugu-6-vowel-dataset` | Telugu |

Bengali (BanglaLekha) will be downloaded automatically from Mendeley.


In [ ]:
# ── 4. Dataset Path Auto-Detection ──────────────────────────────────────────
# Resolve project root regardless of kernel CWD
def _find_local_data():
    for _base in [Path.cwd(), Path.cwd().parent, Path.cwd().parent.parent]:
        if (_base / 'data' / 'raw').exists():
            return _base / 'data' / 'raw'
        if (_base / 'train.ipynb').exists():
            return _base / 'data' / 'raw'
    return Path('./data/raw')
LOCAL = _find_local_data()
print(f'LOCAL = {LOCAL}')

# Devanagari  (UCI dataset slug: medahmedkrichen/devanagari-handwritten-character-datase)
DEVA_TRAIN = find_path(
    '/kaggle/input/devanagari-handwritten-character-datase'
    '/DevanagariHandwrittenCharacterDataset/Train',
    '/kaggle/input/handwritten-devanagari-character-dataset'
    '/DevanagariHandwrittenCharacterDataset/Train',
    str(LOCAL / 'devanagari' / 'DevanagariHandwrittenCharacterDataset' / 'Train'),
    str(LOCAL / 'devanagari' / 'Train'),
)
DEVA_TEST = find_path(
    '/kaggle/input/devanagari-handwritten-character-datase'
    '/DevanagariHandwrittenCharacterDataset/Test',
    '/kaggle/input/handwritten-devanagari-character-dataset'
    '/DevanagariHandwrittenCharacterDataset/Test',
    str(LOCAL / 'devanagari' / 'DevanagariHandwrittenCharacterDataset' / 'Test'),
    str(LOCAL / 'devanagari' / 'Test'),
)

# Tamil  (Kaggle slug: faizalhajamohideen/uthcdtamil-handwritten-database)
def _find_tamil(base):
    if base is None or not base.exists(): return None, None
    for priority in ['85-15', '80-20', '70-30', '90-10']:
        for split_dir in base.iterdir():
            if not split_dir.is_dir(): continue
            if priority in split_dir.name:
                for inner in split_dir.iterdir():
                    if not inner.is_dir(): continue
                    cw_tr = inner / 'train-test-classwise' / 'train'
                    cw_te = inner / 'train-test-classwise' / 'test'
                    if cw_tr.exists(): return cw_tr, cw_te
                    tr, te = inner / 'train', inner / 'test'
                    if tr.exists(): return tr, te
    for cw in sorted(base.rglob('train-test-classwise')):
        tr, te = cw / 'train', cw / 'test'
        if tr.exists(): return tr, te
    trains = sorted(base.rglob('train'))
    tests  = sorted(base.rglob('test'))
    return (trains[0] if trains else None), (tests[0] if tests else None)

TAMIL_BASE = find_path('/kaggle/input/uthcdtamil-handwritten-database',
                        str(LOCAL / 'tamil'))
TAMIL_TRAIN, TAMIL_TEST = _find_tamil(TAMIL_BASE)

# Bengali  (BanglaLekha-Isolated — download from Mendeley if not found)
BENGALI_DIR = find_path(
    '/kaggle/input/banglalekha-isolated/BanglaLekha-Isolated/Images',
    '/kaggle/input/banglalekha-isolated-1/BanglaLekha-Isolated/Images',
    str(LOCAL / 'bengali' / 'BanglaLekha-Isolated' / 'Images'),
    str(LOCAL / 'bengali'),
)
if BENGALI_DIR is None:
    print('Bengali not found — downloading from Mendeley (~163 MB) ...')
    try:
        import urllib.request
        _url     = 'https://data.mendeley.com/public-api/zip/hf6sf8zrkc/download/2'
        _zip     = WORK_DIR / 'BanglaLekha.zip'
        _out_dir = WORK_DIR / 'bengali'
        if not _zip.exists():
            urllib.request.urlretrieve(_url, _zip)
        if not _out_dir.exists():
            with zipfile.ZipFile(_zip) as z:
                z.extractall(_out_dir)
        BENGALI_DIR = find_path(
            str(_out_dir / 'BanglaLekha-Isolated' / 'Images'), str(_out_dir))
        print(f'  Bengali ready: {BENGALI_DIR}')
    except Exception as _e:
        print(f'  Download failed: {_e}  (Bengali will be skipped)')

# Telugu  (Kaggle slug: syamkakarla/telugu-6-vowel-dataset)
TELUGU_CSV = find_path(
    '/kaggle/input/telugu-6-vowel-dataset'
    '/CSV_datasetsix_vowel_dataset_with_class.csv',
    str(LOCAL / 'telugu' / 'CSV_datasetsix_vowel_dataset_with_class.csv'),
)

print('\nDataset status:')
for k, v in [('Devanagari train', DEVA_TRAIN), ('Devanagari test', DEVA_TEST),
             ('Tamil train',      TAMIL_TRAIN), ('Tamil test',     TAMIL_TEST),
             ('Bengali dir',      BENGALI_DIR), ('Telugu CSV',     TELUGU_CSV)]:
    print(f'  [{"OK" if v else "MISSING":7s}] {k}: {v}')


LOCAL = /Users/dhruvrajput/sem-2-assignments/SMAI/A3/data/raw

Dataset status:
  [OK     ] Devanagari train: /Users/dhruvrajput/sem-2-assignments/SMAI/A3/data/raw/devanagari/DevanagariHandwrittenCharacterDataset/Train
  [OK     ] Devanagari test: /Users/dhruvrajput/sem-2-assignments/SMAI/A3/data/raw/devanagari/DevanagariHandwrittenCharacterDataset/Test
  [OK     ] Tamil train: /Users/dhruvrajput/sem-2-assignments/SMAI/A3/data/raw/tamil/uTHCD_c(85-15-split)/85-15-split/train-test-classwise/train
  [OK     ] Tamil test: /Users/dhruvrajput/sem-2-assignments/SMAI/A3/data/raw/tamil/uTHCD_c(85-15-split)/85-15-split/train-test-classwise/test
  [OK     ] Bengali dir: /Users/dhruvrajput/sem-2-assignments/SMAI/A3/data/raw/bengali/BanglaLekha-Isolated/Images
  [OK     ] Telugu CSV: /Users/dhruvrajput/sem-2-assignments/SMAI/A3/data/raw/telugu/CSV_datasetsix_vowel_dataset_with_class.csv


## 5. Exploratory Data Analysis

In [ ]:
# ── 5. EDA — Sample grids ───────────────────────────────────────────────────
def _show_grid(root_or_csv, title, n_cls=5, n_per=4, is_csv=False):
    import random
    from PIL import Image
    fig, axes = plt.subplots(n_cls, n_per, figsize=(n_per * 1.8, n_cls * 1.8))
    if is_csv:
        df   = pd.read_csv(root_or_csv)
        side = int((df.shape[1] - 1) ** 0.5)
        for ci in range(n_cls):
            sub = df[df.iloc[:, -1] == ci]
            rows = sub.sample(min(n_per, len(sub)), random_state=ci)
            for j, (_, row) in enumerate(rows.iterrows()):
                axes[ci, j].imshow(
                    row.values[:-1].reshape(side, -1).astype('uint8'), cmap='gray')
                axes[ci, j].axis('off')
    else:
        cls_dirs = sorted([d for d in Path(root_or_csv).iterdir()
                           if d.is_dir()])[:n_cls]
        for ci, cd in enumerate(cls_dirs):
            imgs = sorted(cd.glob('*.png'))[:n_per]
            for j, ip in enumerate(imgs):
                axes[ci, j].imshow(Image.open(ip).convert('L'), cmap='gray')
                axes[ci, j].axis('off')
            axes[ci, 0].set_ylabel(cd.name[:12], fontsize=7,
                                   rotation=0, labelpad=40)
    fig.suptitle(title, fontsize=11)
    plt.tight_layout()
    out = FIG_DIR / f'eda_{title.lower().replace(" ", "_")}.png'
    plt.savefig(out, dpi=100); plt.show()
    print(f'  Saved {out}')

if DEVA_TRAIN:  _show_grid(DEVA_TRAIN,  'Devanagari Train')
if TAMIL_TRAIN: _show_grid(TAMIL_TRAIN, 'Tamil Train')
if BENGALI_DIR: _show_grid(BENGALI_DIR, 'Bengali')
if TELUGU_CSV:  _show_grid(TELUGU_CSV,  'Telugu CSV', is_csv=True)


  Saved working/figures/eda_devanagari_train.png
  Saved working/figures/eda_tamil_train.png
  Saved working/figures/eda_bengali.png
  Saved working/figures/eda_telugu_csv.png


## 6. Stage 1 — Script Router Training

In [ ]:
# ── 6. Build Router Dataset & Train ─────────────────────────────────────────
print('=' * 60)
print('STAGE 1 — Script Router  (4-class)')
print('=' * 60)

router_tr_dl, router_vl_dl, router_cw = build_router_loaders(
    deva_train  = DEVA_TRAIN,
    tamil_train = TAMIL_TRAIN,
    bengali_dir = BENGALI_DIR,
    telugu_csv  = TELUGU_CSV,
)
print(f'Router train : {len(router_tr_dl.dataset):,} samples  ({len(router_tr_dl)} batches)')
print(f'Router val   : {len(router_vl_dl.dataset):,} samples')
print(f'Class weights: {router_cw.tolist()}\n')

router_model = wrap_model(ScriptRouter(num_classes=4))

router_history, router_best_acc = train_model(
    router_model, router_tr_dl, router_vl_dl,
    tag           = 'router',
    epochs        = ROUTER_EPOCHS,
    patience      = PATIENCE,
    class_weights = router_cw,
)
plot_history(router_history, 'Router')
free_memory()


STAGE 1 — Script Router  (4-class)
Router train : 680 samples  (22 batches)
Router val   : 120 samples
Class weights: [1.0, 1.0, 1.0, 1.0]

  [router]   1/3  tr=78.8%  vl=71.7%  best=71.7%  0m 0s [saved]
  [router]   2/3  tr=97.8%  vl=100.0%  best=100.0%  0m 1s [saved]
  [router] best val acc = 100.00%

  Saved curve → working/figures/curve_Router.png


In [ ]:
# ── 6b. Router Evaluation ────────────────────────────────────────────────────
SCRIPT_NAMES = ['Devanagari', 'Tamil', 'Bengali', 'Telugu']

_, _, r_preds, r_lbls = validate(
    router_model, router_vl_dl, nn.CrossEntropyLoss()
)
present_labels = sorted(set(r_lbls))
present_names  = [SCRIPT_NAMES[i] for i in present_labels]
print('Script Router — Classification Report (val)')
print(classification_report(r_lbls, r_preds,
                             labels=present_labels,
                             target_names=present_names, zero_division=0))

cm = confusion_matrix(r_lbls, r_preds, labels=present_labels)
fig, ax = plt.subplots(figsize=(5, 4))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=present_names, yticklabels=present_names, ax=ax)
ax.set(title='Script Router Confusion Matrix',
       xlabel='Predicted', ylabel='True')
plt.tight_layout()
plt.savefig(FIG_DIR / 'cm_router.png', dpi=120)

plt.show()
print(f'Router best val acc = {router_best_acc:.2f}%')

Script Router — Classification Report (val)
              precision    recall  f1-score   support

  Devanagari       0.97      1.00      0.98        30
       Tamil       1.00      1.00      1.00        30
     Bengali       1.00      0.97      0.98        30
      Telugu       1.00      1.00      1.00        30

    accuracy                           0.99       120
   macro avg       0.99      0.99      0.99       120
weighted avg       0.99      0.99      0.99       120

Router best val acc = 100.00%


## 7. Stage 2 — Character Classifier Training

Trained **sequentially** — each model is freed from GPU memory before the next.
On Kaggle 2×T4, `nn.DataParallel` splits batches across both GPUs automatically.


In [ ]:
# ── 7. Script-Specific Configs ───────────────────────────────────────────────
# CHAR_MAX: max images per class.  50 in smoke run, None (unlimited) in full run.
_MAX = CHAR_MAX

SCRIPT_CFG = [
    {
        'name'        : 'devanagari',
        'num_classes' : 46,
        'num_layers'  : 4,
        'dropout'     : 0.5,
        'batch'       : CHAR_BATCH,
        'train_fn'    : lambda: ScriptImageDataset(
            DEVA_TRAIN, transform=get_transforms('train'),
            max_per_class=_MAX, split='train') if DEVA_TRAIN else None,
        'val_fn'      : lambda: ScriptImageDataset(
            DEVA_TRAIN, transform=get_transforms('val'),
            max_per_class=_MAX, split='val') if DEVA_TRAIN else None,
        'test_fn'     : lambda: ScriptImageDataset(
            DEVA_TEST, transform=get_transforms('val')) if DEVA_TEST else None,
    },
    {
        'name'        : 'tamil',
        'num_classes' : 156,
        'num_layers'  : 4,
        'dropout'     : 0.5,
        'batch'       : CHAR_BATCH,
        'train_fn'    : lambda: ScriptImageDataset(
            TAMIL_TRAIN, transform=get_transforms('train'),
            max_per_class=_MAX, split='train') if TAMIL_TRAIN else None,
        'val_fn'      : lambda: ScriptImageDataset(
            TAMIL_TRAIN, transform=get_transforms('val'),
            max_per_class=_MAX, split='val') if TAMIL_TRAIN else None,
        'test_fn'     : lambda: ScriptImageDataset(
            TAMIL_TEST, transform=get_transforms('val')) if TAMIL_TEST else None,
    },
    {
        'name'        : 'bengali',
        'num_classes' : 84,
        'num_layers'  : 4,
        'dropout'     : 0.5,
        'batch'       : CHAR_BATCH,
        'train_fn'    : lambda: BanglaLekhaDataset(
            BENGALI_DIR, split='train', transform=get_transforms('train'),
            max_per_class=_MAX) if BENGALI_DIR else None,
        'val_fn'      : lambda: BanglaLekhaDataset(
            BENGALI_DIR, split='val', transform=get_transforms('val'),
            max_per_class=_MAX) if BENGALI_DIR else None,
        'test_fn'     : lambda: BanglaLekhaDataset(
            BENGALI_DIR, split='test', transform=get_transforms('val')) if BENGALI_DIR else None,
    },
    {
        'name'        : 'telugu',
        'num_classes' : 6,
        'num_layers'  : 3,
        'dropout'     : 0.6,
        'batch'       : TELUGU_BATCH,
        'train_fn'    : lambda: TeluguCSVDataset(
            TELUGU_CSV, split='train',
            transform=get_transforms('train')) if TELUGU_CSV else None,
        'val_fn'      : lambda: TeluguCSVDataset(
            TELUGU_CSV, split='val',
            transform=get_transforms('val')) if TELUGU_CSV else None,
        'test_fn'     : lambda: TeluguCSVDataset(
            TELUGU_CSV, split='test',
            transform=get_transforms('val')) if TELUGU_CSV else None,
    },
]
print('Script configs:', [c['name'] for c in SCRIPT_CFG])
print(f'CHAR_MAX = {_MAX}  (50 in smoke run, None in full run)')


Script configs: ['devanagari', 'tamil', 'bengali', 'telugu']
CHAR_MAX = 50  (50 in smoke run, None in full run)


In [ ]:
# ── 8. Train All 4 Character Classifiers ─────────────────────────────────────
char_histories = {}
char_best_accs = {}

for cfg in SCRIPT_CFG:
    name = cfg['name']
    print('=' * 60)
    print(f'STAGE 2 — {name.upper()}  ({cfg["num_classes"]} classes)')
    print('=' * 60)

    tr_ds = cfg['train_fn']()
    vl_ds = cfg['val_fn']()
    if tr_ds is None or vl_ds is None:
        print('  SKIP: dataset unavailable\n')
        continue

    print(f'  train={len(tr_ds):,}  val={len(vl_ds):,}  '
          f'num_classes={tr_ds.num_classes}')

    sampler = weighted_sampler(tr_ds) if name == 'tamil' else None
    tr_dl   = make_loader(tr_ds, cfg['batch'],
                          shuffle=(sampler is None), sampler=sampler)
    vl_dl   = make_loader(vl_ds, cfg['batch'], shuffle=False)
    cw      = tr_ds.class_weights()

    model = wrap_model(ScriptCNN(
        num_classes = cfg['num_classes'],
        num_layers  = cfg['num_layers'],
        dropout     = cfg['dropout'],
    ))
    hist, best = train_model(
        model, tr_dl, vl_dl,
        tag           = name,
        epochs        = CHAR_EPOCHS,
        patience      = PATIENCE,
        class_weights = cw,
    )
    char_histories[name] = hist
    char_best_accs[name] = best
    plot_history(hist, name)

    del model, tr_ds, vl_ds, tr_dl, vl_dl, cw
    free_memory()

print('\nAll classifiers trained.')
print({k: f'{v:.2f}%' for k, v in char_best_accs.items()})


STAGE 2 — DEVANAGARI  (46 classes)
  train=1,932  val=368  num_classes=46
  [devanagari]   1/3  tr=5.8%  vl=26.6%  best=26.6%  0m 2s [saved]
  [devanagari]   2/3  tr=18.6%  vl=50.3%  best=50.3%  0m 3s [saved]
  [devanagari]   3/3  tr=30.1%  vl=61.7%  best=61.7%  0m 5s [saved]
  [devanagari] best val acc = 61.68%

  Saved curve → working/figures/curve_devanagari.png
STAGE 2 — TAMIL  (156 classes)
  train=6,552  val=1,248  num_classes=156
  [tamil]   1/3  tr=5.2%  vl=25.3%  best=25.3%  0m 5s [saved]
  [tamil]   2/3  tr=18.1%  vl=40.5%  best=40.5%  0m 10s [saved]
  [tamil]   3/3  tr=28.3%  vl=61.9%  best=61.9%  0m 15s [saved]
  [tamil] best val acc = 61.86%

  Saved curve → working/figures/curve_tamil.png
STAGE 2 — BENGALI  (84 classes)
  train=4,200  val=4,200  num_classes=84
  [bengali]   1/3  tr=5.0%  vl=18.6%  best=18.6%  0m 6s [saved]
  [bengali]   2/3  tr=13.5%  vl=34.1%  best=34.1%  0m 11s [saved]
  [bengali]   3/3  tr=20.1%  vl=42.6%  best=42.6%  0m 15s [saved]
  [bengali] best va

## 8. Evaluation

In [ ]:
# ── 9. Per-Script Evaluation ─────────────────────────────────────────────────
all_metrics = {}

for cfg in SCRIPT_CFG:
    name = cfg['name']
    print(f'\nEvaluating: {name.upper()}')
    metrics, preds, lbls = eval_script(name, cfg)
    if metrics is None:
        continue
    all_metrics[name] = metrics
    print(f'  Top-1      : {metrics["top1_acc"]:.2f}%')
    print(f'  Top-5      : {metrics["top5_acc"]:.2f}%')
    print(f'  Macro F1   : {metrics["macro_f1"]:.2f}')
    plot_confusion(preds, lbls, name)

print('\n', all_metrics)



Evaluating: DEVANAGARI
  Top-1      : 59.70%
  Top-5      : 89.14%
  Macro F1   : 57.51
  Saved confusion matrix → working/figures/cm_devanagari.png

Evaluating: TAMIL
  Top-1      : 55.94%
  Top-5      : 86.84%
  Macro F1   : 54.26
  Saved confusion matrix → working/figures/cm_tamil.png

Evaluating: BENGALI
  Top-1      : 41.95%
  Top-5      : 73.86%
  Macro F1   : 38.38
  Saved confusion matrix → working/figures/cm_bengali.png

Evaluating: TELUGU
  Top-1      : 95.00%
  Top-5      : 100.00%
  Macro F1   : 94.84
  Saved confusion matrix → working/figures/cm_telugu.png

 {'devanagari': {'script': 'devanagari', 'num_classes': 46, 'test_size': 13800, 'top1_acc': 59.7, 'top5_acc': 89.14, 'macro_f1': 57.51, 'weighted_f1': 57.51}, 'tamil': {'script': 'tamil', 'num_classes': 156, 'test_size': 13574, 'top1_acc': 55.94, 'top5_acc': 86.84, 'macro_f1': 54.26, 'weighted_f1': 54.01}, 'bengali': {'script': 'bengali', 'num_classes': 84, 'test_size': 24916, 'top1_acc': 41.95, 'top5_acc': 73.86, 'mac

In [ ]:
# ── 9b. Robustness + Latency benchmark (skipped in smoke/test run) ───────────
robust_results  = {}
latency_results = {}

if not (KAGGLE_SMOKE or TEST_RUN):
    robust_results  = run_robustness_eval(SCRIPT_CFG)
    plot_robustness(robust_results)
    latency_results = run_latency_benchmark(SCRIPT_CFG)
else:
    print('Robustness / latency eval skipped in smoke/test run.')


Robustness / latency eval skipped in smoke/test run.


## 9. Ablation Studies

7 studies, each varying **one factor** on Devanagari (full run only).


In [ ]:
# ── 10. Ablation Studies ─────────────────────────────────────────────────────
ABLATION_RESULTS = {}

if RUN_ABLATIONS and DEVA_TRAIN and DEVA_TEST:
    print('=' * 60)
    print('ABLATION STUDIES — Devanagari (46 classes)')
    print('=' * 60)
    ABLATION_RESULTS = run_all_ablations(
        deva_train    = DEVA_TRAIN,
        deva_test     = DEVA_TEST,
        max_per_class = 300,
    )
    plot_ablation_results(ABLATION_RESULTS)
else:
    print('Ablations skipped (smoke/test run or data unavailable).')


Ablations skipped (smoke/test run or data unavailable).


## 10. Results Export

In [ ]:
# ── 11. Save Results & Final Summary ────────────────────────────────────────
final_results = {
    'env'                  : ENV,
    'kaggle_smoke'         : KAGGLE_SMOKE,
    'img_size'             : IMG_SIZE,
    'router_best_val_acc'  : globals().get('router_best_acc'),
    'classifiers'          : all_metrics,
    'robustness'           : robust_results,
    'latency'              : latency_results,
    'ablations'            : ABLATION_RESULTS,
}
out_path = RES_DIR / 'final_results.json'
with open(out_path, 'w') as f:
    json.dump(final_results, f, indent=2, default=str)
print(f'Results saved to {out_path}')

print('\n' + '=' * 60)
print('FINAL SUMMARY')
print('=' * 60)
print(f'  Router best val acc : {globals().get("router_best_acc", "N/A")}')
for k, v in char_best_accs.items():
    print(f'  {k:<20}: {v:.2f}%')
print('=' * 60)


Results saved to working/results/final_results.json

FINAL SUMMARY
  Router best val acc : 100.0
  devanagari          : 61.68%
  tamil               : 61.86%
  bengali             : 42.62%
  telugu              : 88.33%
